# Notebook 06 — Phase 2B : Fine-tuning XLM-RoBERTa
**Projet: SentimentsLoupe**

---
## Objectif

1. Charger **XLM-RoBERTa** depuis Hugging Face (modèle multilingue 100 langues)
2. Fine-tuning sur Flipkart (7700) + Trustpilot (1235) = **8929 avis**
3. Test **zero-shot** sur Jumia (60 avis FR/Darija)
4. **Comparer** avec le modèle TF-IDF Phase 1

> XLM-RoBERTa comprend le français et l'arabe sans entraînement spécifique → idéal pour Jumia

---
## 0. Installation des dépendances (1ère exécution seulement)

In [ ]:
import sys, warnings, json
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import torch
import joblib
from pathlib import Path

from sklearn.metrics import f1_score, accuracy_score
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from datasets import Dataset

from src.config.settings import MODELS_DIR, RAW_DIR, PROCESSED_DIR
from src.preprocessing.source_normalizer import SourceNormalizer
from src.preprocessing.feature_engineering import FeatureEngineer

print('✅ Setup OK')

---
## 1. Chargement des données

In [ ]:
# Charger les artefacts Phase 1
champion_obj = joblib.load(str(MODELS_DIR / 'champion_model.pkl'))

with open(str(MODELS_DIR / 'phase1_metrics.json'), 'r') as f:
    metrics_p1 = json.load(f)

F1_REF = metrics_p1['champion_metrics']['f1_macro']
print(f'Modèle TF-IDF Phase 1: F1 = {F1_REF:.4f}')

# Charger les sources
# Flipkart
df_fk = pd.read_csv(str(PROCESSED_DIR / 'flipkart_cleaned.csv'))
print(f'Flipkart: {len(df_fk)} avis')

# Trustpilot
df_tp = pd.read_parquet(str(RAW_DIR / 'trustpilot_multidomain.parquet'))
if 'text' in df_tp.columns and 'review' not in df_tp.columns:
    df_tp = df_tp.rename(columns={'text': 'review'})
if 'rating' not in df_tp.columns and 'score' in df_tp.columns:
    df_tp = df_tp.rename(columns={'score': 'rating'})
df_tp = df_tp[df_tp['rating'].between(1, 5)][['review', 'rating']].copy()
print(f'Trustpilot: {len(df_tp)} avis')

# Jumia
df_jm = pd.read_parquet(str(RAW_DIR / 'jumia_maroc.parquet'))
print(f'Jumia: {len(df_jm)} avis')

---
## 2. Préparation des labels

In [ ]:
SENTIMENT_MAP = {1: 'negative', 2: 'negative', 3: 'neutral', 4: 'positive', 5: 'positive'}
LABEL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}
ID2LABEL = {0: 'negative', 1: 'neutral', 2: 'positive'}

# Préparer dataset d'entraînement (Flipkart + Trustpilot)
# On garde les textes ORIGINAUX (pas nettoyés) pour XLM-RoBERTa, il sait gérer les accents
df_fk_train = df_fk[['review', 'rating']].copy()
df_fk_train['sentiment'] = df_fk_train['rating'].map(SENTIMENT_MAP)
df_fk_train['label'] = df_fk_train['sentiment'].map(LABEL2ID)

df_tp['sentiment'] = df_tp['rating'].map(SENTIMENT_MAP)
df_tp['label'] = df_tp['sentiment'].map(LABEL2ID)

df_train = pd.concat([df_fk_train, df_tp[['review', 'rating', 'sentiment', 'label']]], ignore_index=True)
df_train = df_train.dropna(subset=['label']).copy()
print(f'Dataset entraînement: {len(df_train)} avis')
print(f'Distribution: {df_train["sentiment"].value_counts().to_dict()}')

In [ ]:
# Préparer Jumia (test)
df_jm['review'] = df_jm['text']  # Renommer 'text' en 'review'
df_jm['sentiment'] = df_jm['rating'].map(SENTIMENT_MAP)
df_jm['label'] = df_jm['sentiment'].map(LABEL2ID)
df_test_jm = df_jm.dropna(subset=['label']).copy()
print(f'Dataset test Jumia: {len(df_test_jm)} avis')
print(f'Colonnes Jumia: {list(df_test_jm.columns)}')

---
## 3. Tokenization

In [ ]:
MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 64   # Réduit de 128 à 64 (plus rapide)
BATCH_SIZE = 32   # Doublé (plus de mémoire mais plus rapide)
EPOCHS = 1        # 1 epoch suffit pour un premier test

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Tokenizer chargé: {MODEL_NAME}')
print(f'Taille vocabulaire: {tokenizer.vocab_size}')

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples['review'],
        padding='max_length',
        truncation=True,
        max_length=MAX_LENGTH
    )

# Convertir en Dataset Hugging Face
dataset_train = Dataset.from_pandas(df_train[['review', 'label']])
dataset_test_jm = Dataset.from_pandas(df_test_jm[['review', 'label']])

# Tokenizer
dataset_train = dataset_train.map(tokenize_function, batched=True)
dataset_test_jm = dataset_test_jm.map(tokenize_function, batched=True)

# Format PyTorch
dataset_train = dataset_train.with_format('torch', columns=['input_ids', 'attention_mask', 'label'])
dataset_test_jm = dataset_test_jm.with_format('torch', columns=['input_ids', 'attention_mask', 'label'])

# Split train/validation
split = dataset_train.train_test_split(test_size=0.1, seed=42)
dataset_val = split['test']
dataset_train = split['train']

print(f'Train: {len(dataset_train)} | Validation: {len(dataset_val)}')
print(f'Test Jumia: {len(dataset_test_jm)}')

---
## 4. Fine-tuning XLM-RoBERTa

In [ ]:
# Charger le modèle
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)
print(f'Modèle chargé: {MODEL_NAME}')
print(f'Paramètres: {model.num_parameters():,}')

In [ ]:
# ── CHARGER LE MODÈLE FINE-TUNÉ SAUVEGARDÉ ──
# Le modèle a déjà été entraîné et sauvegardé dans data/models/xlm_roberta_finetuned/
# On charge directement les poids, pas besoin de ré-entraîner.

OUTPUT_DIR = MODELS_DIR / 'xlm_roberta_finetuned'

# Charger le modèle et tokenizer sauvegardés
model = AutoModelForSequenceClassification.from_pretrained(
    str(OUTPUT_DIR),
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)
tokenizer = AutoTokenizer.from_pretrained(str(OUTPUT_DIR))

print(f'✅ Modèle chargé depuis: {OUTPUT_DIR}')
print(f'   Paramètres: {model.num_parameters():,}')
print(f'   → Pas besoin de ré-entraîner !')

# Re-créer le Trainer avec le modèle chargé (pour pouvoir faire trainer.predict)
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_eval_batch_size=BATCH_SIZE,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
)

print('✅ Trainer prêt pour la prédiction !')

---
## 5. Évaluation sur Jumia

In [ ]:
# ── Test sur Trustpilot ──
# Préparer Trustpilot en dataset Hugging Face
df_tp_test = df_tp[['review', 'label']].dropna(subset=['label']).copy()
dataset_test_tp = Dataset.from_pandas(df_tp_test)
dataset_test_tp = dataset_test_tp.map(tokenize_function, batched=True)
dataset_test_tp = dataset_test_tp.with_format('torch', columns=['input_ids', 'attention_mask', 'label'])

# Prédiction sur Trustpilot
pred_tp = trainer.predict(dataset_test_tp)
y_pred_tp = np.argmax(pred_tp.predictions, axis=-1)
y_true_tp = dataset_test_tp.with_format('numpy')['label']

f1_tp_xlm = f1_score(y_true_tp, y_pred_tp, average='macro')
acc_tp_xlm = accuracy_score(y_true_tp, y_pred_tp)
print(f'\n📊 XLM-RoBERTa sur Trustpilot (même source que train):')
print(f'F1-Macro: {f1_tp_xlm:.4f}')
print(f'Accuracy: {acc_tp_xlm:.4f}')

# ── Test sur Jumia ──
predictions = trainer.predict(dataset_test_jm)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = dataset_test_jm.with_format('numpy')['label']

f1_jm_xlm = f1_score(y_true, y_pred, average='macro')
acc_jm_xlm = accuracy_score(y_true, y_pred)
print(f'\n📊 XLM-RoBERTa sur Jumia (langue différente):')
print(f'F1-Macro: {f1_jm_xlm:.4f}')
print(f'Accuracy: {acc_jm_xlm:.4f}')

---
## 6. Comparaison finale

In [ ]:
# ── Charger les scores des modèles précédents ──
# Score Phase 1 (Flipkart seul, Notebook 03)
F1_PHASE1 = F1_REF  # = 0.6491 sur Flipkart test

# Scores Notebook 05 (obtenus précédemment)
F1_TP_P1 = 0.4297   # Zero-Shot Trustpilot (Phase 1)
F1_JUMIA_P1 = 0.3395  # Zero-Shot Jumia (Phase 1)
F1_JUMIA_P2A = 0.3764 # Amélioration FK+TP sur Jumia

print("\n" + "=" * 95)
print("  🏆 TABLEAU COMPARATIF FINAL — SentimentsLoupe")
print("=" * 95)

print(f"\n{'Modèle':<35} {'Entraîné sur':<20} {'Trustpilot F1':<15} {'Jumia F1':<12} {'Gain vs P1 (Jumia)'}")
print("-" * 100)
print(f"{'Phase 1 — TF-IDF champion':<35} {'Flipkart (7K)':<20} {F1_TP_P1:<15.4f} {F1_JUMIA_P1:<12.4f} {'—':<18}")
print(f"{'Phase 2A — TF-IDF amélioré':<35} {'FK+Trustpilot (9K)':<20} {'(dans train)':<15} {F1_JUMIA_P2A:<12.4f} {F1_JUMIA_P2A - F1_JUMIA_P1:<+18.4f}")
print(f"{'Phase 2B — XLM-RoBERTa 🚀':<35} {'FK+Trustpilot (9K)':<20} {f1_tp_xlm:<15.4f} {f1_jm_xlm:<12.4f} {f1_jm_xlm - F1_JUMIA_P1:<+18.4f}")

---
## 7. Sauvegarde du modèle

In [ ]:
# Sauvegarde du modèle fine-tuné
# model.save_pretrained(str(OUTPUT_DIR))
# tokenizer.save_pretrained(str(OUTPUT_DIR))
# print(f'✅ Modèle sauvegardé: {OUTPUT_DIR}')

---
## 8. Instructions d'exécution

1. **Installer les dépendances** (1ère fois seulement) :
   ```powershell
   pip install torch --index-url https://download.pytorch.org/whl/cpu
   pip install transformers datasets evaluate accelerate
   ```

2. **Activer le venv** : `\venv\Scripts\Activate.ps1`

3. **Exécuter les cellules dans l'ordre** jusqu'à la section 4

4. **Lancer l'entraînement** en décommentant : `trainer.train()`

5. **Évaluer sur Jumia** dans la section 5

6. **Sauvegarder** le modèle fine-tuné